# 🔭 Laboratorio: Macro-Backtesting de Validación Secuencial
**Objetivo:** Ejecutar simulaciones `Walk-Forward` masivas sobre múltiples activos y marcos temporales para crear una base de datos de rendimiento empírico del motor SINDy.
**Métricas:** RMSE y Hit Ratio (Direccionalidad).
**Archivo de Salida:** `macro_backtest_db.csv` (Anexión Segura / Checkpointing).

In [2]:
import sys
import os
import time
import warnings
sys.path.append(os.path.abspath('..'))

# Ocultar warnings matemáticos durante la simulación masiva
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from src.ui.market_loader import MarketLoader
from src.quant_engine.evaluator import WalkForwardEvaluator

### 1. Configuración de la Batería de Pruebas
Aquí se define el universo de activos y las temporalidades a auditar.

In [3]:
# 1. Definir los Tickers a analizar
tickers = ['MSFT', 'XLF', 'C', 'BTC-USD', 'SPY', 'AAPL', 'ETH-USD', 'JPM', 'BAC', 'MA', 'AMZN' ]

# 2. Definir los tamaños de Ventana de Contexto (lookback en velas diarias)
# 60 ≈ 3 meses, 120 ≈ 6 meses, 252 ≈ 1 año, 500 ≈ 2 años, 750 ≈ 3 años, 1250 ≈ 5 años, 0 = Todo el historial (Legacy)
context_windows = [1825]

db_path = "paper_kinetopus.csv"

# 3. Manejador de DB y Checkpointing (Idempotencia)
processed_configs = set()
if os.path.exists(db_path):
    df_existente = pd.read_csv(db_path)
    if not df_existente.empty:
        # Agrupar por Ticker y Context_Window para saber qué combinaciones ya terminaron
        agrupado = df_existente.groupby(['Ticker', 'Context_Window']).size().reset_index()
        for _, row in agrupado.iterrows():
            processed_configs.add((row['Ticker'], int(row['Context_Window'])))

print(f"⚙️ Total Combinaciones Posibles en el Universo: {len(tickers) * len(context_windows)}")
print(f"📂 Combinaciones ya procesadas y guardadas en DB: {len(processed_configs)}\n")


⚙️ Total Combinaciones Posibles en el Universo: 11
📂 Combinaciones ya procesadas y guardadas en DB: 0



### 2. Motor de Ejecución Masiva
Este proceso puede tardar horas. Si lo detienes, la próxima vez que lo inicies continuará donde se quedó.

In [4]:
for ticker in tickers:
    for cw in context_windows:
        config_key = (ticker, cw)
        
        if config_key in processed_configs:
            print(f"⏭️ Omitiendo {ticker} [Contexto: {cw}] - Ya existe en la base de datos.")
            continue
            
        print(f"\n🚀 Iniciando simulación para {ticker} [Contexto: {cw}]...")
        
        # --- DESCARGA CON PROTECCIÓN ---
        try:
            df_mercado = MarketLoader.load_ticker_data(ticker, period="10y", interval="1d")
        except Exception as e:
            print(f"❌ Error descargando datos para {ticker}: {e}")
            continue
            
        total_velas = len(df_mercado)
        if total_velas < 200:
            print(f"⚠️ Omitiendo {ticker}: Historial demasiado corto ({total_velas} velas).")
            continue
            
        # --- LÓGICA DE SALTOS DINÁMICOS ---
        SALTO = 20
        VENTANA_INICIAL = 150
        HORIZONTE = 300
        BLOQUES = 60
        
        print(f"   ► Velas Disponibles: {total_velas} | Salto Iterativo: {SALTO} | Predicción a Ciegas: {HORIZONTE}")
        
        # Instanciamos el Motor con la ventana de contexto
        evaluador = WalkForwardEvaluator(df_mercado, disable_norm=False, disable_returns=False, context_window=cw)
        
        try:
            # Ejecutar el Auto-Tuner Iterativo en el tiempo
            df_resultados = evaluador.run(initial_window=VENTANA_INICIAL, stride=SALTO, horizon=HORIZONTE, blocks=BLOQUES)
            
            # Extraer resultados brutos
            df_guardar = df_resultados.copy()
            
            # INYECCIÓN DE METADATOS PARA EL FUTURO QUERY NOTEBOOK
            df_guardar.insert(0, 'Context_Window', cw)
            df_guardar.insert(0, 'Total_Velas_Disponible', total_velas)
            df_guardar.insert(0, 'Intervalo_Velas', '1d')
            df_guardar.insert(0, 'Periodo_Historia', '10y')
            df_guardar.insert(0, 'Ticker', ticker)
            
            # Checkpoint Progressivo al CSV
            file_exists = os.path.isfile(db_path)
            df_guardar.to_csv(db_path, mode='a', header=not file_exists, index=False)
            
            print(f"✅ Éxito. Guardadas {len(df_guardar)} iteraciones para {ticker} en {db_path}.")
            
            # Registrar para que no se repita en caso de caída posterior en este mismo loop
            processed_configs.add(config_key)
            
            # Breve pausa para limpiar I/O
            time.sleep(1)
            
        except Exception as e:
            print(f"❌ Error matemático/sistémico durante el Backtest de {ticker}: {e}")
            continue



🚀 Iniciando simulación para MSFT [Contexto: 1825]...
   ► Velas Disponibles: 2513 | Salto Iterativo: 20 | Predicción a Ciegas: 300
Iniciando Walk-Forward (Ventana:150, Salto:20, Horizonte:300 velas en 60 bloques, Contexto:1825)


Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando itera

✅ Éxito. Guardadas 104 iteraciones para MSFT en paper_kinetopus.csv.

🚀 Iniciando simulación para XLF [Contexto: 1825]...
   ► Velas Disponibles: 2513 | Salto Iterativo: 20 | Predicción a Ciegas: 300
Iniciando Walk-Forward (Ventana:150, Salto:20, Horizonte:300 velas en 60 bloques, Contexto:1825)


Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.00). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando itera

✅ Éxito. Guardadas 104 iteraciones para XLF en paper_kinetopus.csv.

🚀 Iniciando simulación para C [Contexto: 1825]...
   ► Velas Disponibles: 2513 | Salto Iterativo: 20 | Predicción a Ciegas: 300
Iniciando Walk-Forward (Ventana:150, Salto:20, Horizonte:300 velas en 60 bloques, Contexto:1825)


...

✅ Éxito. Guardadas 104 iteraciones para BAC en paper_kinetopus.csv.

🚀 Iniciando simulación para MA [Contexto: 1825]...
   ► Velas Disponibles: 2513 | Salto Iterativo: 20 | Predicción a Ciegas: 300
Iniciando Walk-Forward (Ventana:150, Salto:20, Horizonte:300 velas en 60 bloques, Contexto:1825)


Física Débil/Inexistente (R2=0.04). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.01). Evitando iteración Monte Carlo por Alucinación Matemática.
Falso Positivo Estocástico Detectado: SINDy anuló los Retornos (dr/dt = 0). Bloqueando cono divergente.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.03). Evita

✅ Éxito. Guardadas 104 iteraciones para MA en paper_kinetopus.csv.

🚀 Iniciando simulación para AMZN [Contexto: 1825]...
   ► Velas Disponibles: 2513 | Salto Iterativo: 20 | Predicción a Ciegas: 300
Iniciando Walk-Forward (Ventana:150, Salto:20, Horizonte:300 velas en 60 bloques, Contexto:1825)


Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando iteración Monte Carlo por Alucinación Matemática.
Física Débil/Inexistente (R2=0.02). Evitando itera

✅ Éxito. Guardadas 104 iteraciones para AMZN en paper_kinetopus.csv.


### 3. Sanidad del Dataset
Lectura rápida para asegurar que la DB se está llenando correctamente.

In [5]:
if os.path.exists(db_path):
    df_final = pd.read_csv(db_path)
    print(f"📊 Total Filas en la Base de Datos: {len(df_final)}")
    display(df_final.head())
    display(df_final.tail())
else:
    print("El archivo CSV aún no ha sido creado.")

📊 Total Filas en la Base de Datos: 1235


,Ticker,Periodo_Historia,Intervalo_Velas,Total_Velas_Disponible,Context_Window,Iteracion (Velas Vistas),Drift (k),SINDy R2,Validez,MAPE_B1,...,RMSE_B59,Hit_B59,CumHit_B59,Profit_B59,MAPE_B60,Naive_MAPE_B60,RMSE_B60,Hit_B60,CumHit_B60,Profit_B60
0,MSFT,10y,1d,2513,1825,150,0.1,0.499561,OK,0.36,...,89.8098,0.0,0.0,-55.56,99.95,35.61,90.9545,0.0,0.0,-55.85
1,MSFT,10y,1d,2513,1825,170,0.1,0.497885,OK,3.21,...,93.1201,0.0,0.0,-58.33,99.84,35.35,91.1493,1.0,0.0,-54.40
2,MSFT,10y,1d,2513,1825,190,0.2,0.504826,OK,0.87,...,1496.5205,1.0,1.0,60.01,1741.30,38.92,1751.2848,0.0,1.0,62.14
3,MSFT,10y,1d,2513,1825,210,0.3,0.561431,OK,0.58,...,99.8426,1.0,0.0,-57.29,99.75,36.18,99.1530,0.0,0.0,-58.49
4,MSFT,10y,1d,2513,1825,230,0.3,0.497734,OK,0.44,...,804.9828,1.0,1.0,56.47,902.26,36.84,950.9060,1.0,1.0,60.00


,Ticker,Periodo_Historia,Intervalo_Velas,Total_Velas_Disponible,Context_Window,Iteracion (Velas Vistas),Drift (k),SINDy R2,Validez,MAPE_B1,...,RMSE_B59,Hit_B59,CumHit_B59,Profit_B59,MAPE_B60,Naive_MAPE_B60,RMSE_B60,Hit_B60,CumHit_B60,Profit_B60
1230,AMZN,10y,1d,2513,1825,2130,1.2,0.921093,OK,2.39,...,55.7789,1.0,1.0,0.94,12.75,4.57,31.7127,1.0,1.0,6.09
1231,AMZN,10y,1d,2513,1825,2150,0.8,0.999918,OK,2.12,...,86.9195,0.0,0.0,-35.86,30.54,24.90,81.6644,1.0,0.0,-31.61
1232,AMZN,10y,1d,2513,1825,2170,0.8,0.999850,OK,3.80,...,101.2529,1.0,0.0,-37.45,38.44,26.16,92.9778,0.0,0.0,-37.90
1233,AMZN,10y,1d,2513,1825,2190,0.8,0.999800,OK,1.60,...,69.3712,0.0,0.0,-28.23,29.96,23.51,74.5433,0.0,0.0,-34.20
1234,AMZN,10y,1d,2513,1825,2210,0.8,0.999812,OK,1.83,...,74.2851,0.0,0.0,-31.94,28.30,24.26,77.3282,1.0,0.0,-29.34
